# Phase 2 TFIM-QRC v1.5 Readout Robustness Probe

The best v1.5 reservoir improved all test metrics but remains far from the classical ESN baseline. This notebook does not run new reservoir architectures. It reuses the best v1.5 configuration and tests whether the readout is being hurt by noisy/shifted reservoir features.

Tested readout controls:

- train-only feature winsorization at 1/99 percentiles;
- train-only top-k feature selection by absolute feature-target correlation;
- ridge alpha sweep.

This is cheap compared with new QRC simulations and should be interpreted as readout robustness, not a new QRC architecture.

In [ ]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir("..")

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

from qpitome_qrc.data.features import FEATURE_COLUMNS
from qpitome_qrc.data.loaders import load_phase2_volatility_data
from qpitome_qrc.data.pca import fit_transform_pca_splits_train_only
from qpitome_qrc.data.splits import chronological_tabular_split
from qpitome_qrc.evaluation.metrics import evaluate_volatility_forecast
from qpitome_qrc.qrc.tfim_reservoir import (
    TFIMQRCConfig,
    _safe_feature_target_correlations,
    diagnose_reservoir_feature_splits,
    fit_tfim_qrc_regressor,
    make_qrc_sequence_splits,
    summarize_qrc_result,
)

## 1. Data and best v1.5 reservoir run

This recomputes the best v1.5 run once so the notebook is self-contained.

In [ ]:
target = "future_rv_20d"

df = load_phase2_volatility_data()
splits = chronological_tabular_split(df)

pca6 = fit_transform_pca_splits_train_only(
    splits,
    feature_columns=FEATURE_COLUMNS,
    target_columns=[target],
    n_components=6,
    prefix="pca6",
)

sequence_splits_6 = make_qrc_sequence_splits(
    pca6.splits,
    feature_columns=pca6.feature_columns,
    target_column=target,
    lookback_days=40,
)

best_config = TFIMQRCConfig(
    qubits=6,
    pca_components=6,
    lookback_days=40,
    anchor_count=6,
    anchor_policy="even",
    observable_mode="zxzz",
    collect_anchor_features=True,
    topology="full",
    trotter_steps_per_anchor=3,
    virtual_nodes_per_anchor=3,
    coupling_scale=0.7,
    transverse_field=0.5,
    evolution_time=0.5,
    angle_max=np.pi / 2,
    ridge_alpha=3000.0,
    target_transform="log",
    seed=42,
    use_disorder=True,
    disorder_strength=0.20,
)

best_result = fit_tfim_qrc_regressor(
    sequence_splits_6,
    config=best_config,
    target=target,
    verbose=True,
)

pd.DataFrame([summarize_qrc_result(best_result)]).T

## 2. Helper functions

In [ ]:
def winsorize_train_bounds(H_train, lower_pct=1.0, upper_pct=99.0):
    lower = np.percentile(H_train, lower_pct, axis=0)
    upper = np.percentile(H_train, upper_pct, axis=0)
    return lower, upper


def apply_winsor(H, lower, upper):
    return np.clip(H, lower, upper)


def fit_predict_readout(H_train, y_train, H_val, H_test, *, alpha):
    scaler = StandardScaler()
    H_train_scaled = scaler.fit_transform(H_train)
    H_val_scaled = scaler.transform(H_val)
    H_test_scaled = scaler.transform(H_test)

    y_fit = np.log(np.maximum(y_train, 1e-8))
    model = Ridge(alpha=alpha)
    model.fit(H_train_scaled, y_fit)

    return (
        np.exp(model.predict(H_train_scaled)),
        np.exp(model.predict(H_val_scaled)),
        np.exp(model.predict(H_test_scaled)),
    )


def metrics_row(y_train, y_val, y_test, pred_train, pred_val, pred_test):
    train_m = evaluate_volatility_forecast(y_train, pred_train)
    val_m = evaluate_volatility_forecast(y_val, pred_val)
    test_m = evaluate_volatility_forecast(y_test, pred_test)
    return {
        "train_rmse": train_m.rmse,
        "val_rmse": val_m.rmse,
        "test_rmse": test_m.rmse,
        "train_qlike": train_m.qlike,
        "val_qlike": val_m.qlike,
        "test_qlike": test_m.qlike,
        "train_mz_r2": train_m.mz_r2,
        "val_mz_r2": val_m.mz_r2,
        "test_mz_r2": test_m.mz_r2,
    }

## 3. Readout robustness sweep

All feature selection and clipping is fit on the training split only.

In [ ]:
H_train = best_result.train_features
H_val = best_result.val_features
H_test = best_result.test_features

_, y_train, _ = sequence_splits_6["train"]
_, y_val, _ = sequence_splits_6["val"]
_, y_test, _ = sequence_splits_6["test"]

rows = []
diag_rows = []

for winsor in [False, True]:
    if winsor:
        lower, upper = winsorize_train_bounds(H_train, 1.0, 99.0)
        Hw_train = apply_winsor(H_train, lower, upper)
        Hw_val = apply_winsor(H_val, lower, upper)
        Hw_test = apply_winsor(H_test, lower, upper)
    else:
        Hw_train, Hw_val, Hw_test = H_train, H_val, H_test

    corr = _safe_feature_target_correlations(Hw_train, y_train)
    order = np.argsort(np.abs(corr))

    for top_k in [20, 40, 80, 120, H_train.shape[1]]:
        idx = order[-top_k:]
        Hs_train = Hw_train[:, idx]
        Hs_val = Hw_val[:, idx]
        Hs_test = Hw_test[:, idx]

        for alpha in [1000.0, 3000.0, 10000.0, 30000.0]:
            pred_train, pred_val, pred_test = fit_predict_readout(
                Hs_train, y_train, Hs_val, Hs_test, alpha=alpha
            )
            row = metrics_row(y_train, y_val, y_test, pred_train, pred_val, pred_test)
            row.update({
                "winsor": winsor,
                "top_k": top_k,
                "alpha": alpha,
                "n_features": Hs_train.shape[1],
            })
            rows.append(row)

        diag = diagnose_reservoir_feature_splits(
            Hs_train,
            Hs_val,
            Hs_test,
            y_train,
            y_val,
            y_test,
        )
        diag.insert(0, "winsor", winsor)
        diag.insert(1, "top_k", top_k)
        diag_rows.append(diag)

readout_sweep = pd.DataFrame(rows)
readout_diagnostics = pd.concat(diag_rows, ignore_index=True)

readout_sweep.sort_values(["test_rmse", "test_qlike"], ascending=[True, True]).head(20)

## 4. Validation-first view

This table avoids selecting only by test. If validation and test disagree sharply, treat the result as unstable.

In [ ]:
readout_sweep.sort_values(["val_rmse", "test_rmse"], ascending=[True, True]).head(20)

## 5. Diagnostics for selected feature sets

In [ ]:
diagnostic_cols = [
    "winsor",
    "top_k",
    "split",
    "n_samples",
    "n_features",
    "near_constant_features",
    "feature_std_min",
    "feature_std_median",
    "feature_std_max",
    "effective_rank",
    "condition_number",
    "mean_abs_feature_target_corr",
    "max_abs_feature_target_corr",
    "mean_abs_shift_vs_train",
    "max_abs_shift_vs_train",
]

readout_diagnostics[diagnostic_cols].sort_values(["winsor", "top_k", "split"])

## 6. Save outputs

In [ ]:
out_dir = Path("results/tables")
out_dir.mkdir(parents=True, exist_ok=True)

readout_sweep.to_csv(out_dir / "phase2_tfim_qrc_v15_readout_robustness_probe.csv", index=False)
readout_diagnostics.to_csv(out_dir / "phase2_tfim_qrc_v15_readout_robustness_diagnostics.csv", index=False)

print("Saved readout robustness outputs to", out_dir)

## 7. Reference numbers

```text
v1 best static QRC:
RMSE  = 0.102618
QLIKE = -1.942716
MZ R² = 0.072134

v1.5 best reservoir before readout robustness:
RMSE  = 0.102084
QLIKE = -2.033862
MZ R² = 0.080576

Classical ESN baseline remains substantially better:
RMSE ≈ 0.075–0.086
MZ R² ≈ 0.45–0.53
```